# 03 — Quantize & Benchmark All Methods

Apply GPTQ, AWQ, BnB-NF4, and BnB-INT8 to the merged fine-tuned Gemma 2 2B model.
Benchmark each variant on all metrics.

**Running on Kaggle T4 16GB:**
- BnB NF4/INT8: loads directly in quantized form — fast and easy
- GPTQ: uses `device_map="auto"` — Gemma 2 2B fits easily on T4
- AWQ: same approach
- Calibration data: 64 medical text samples from PubMedQA

In [ ]:
!pip install -q "transformers>=5.5.0" peft bitsandbytes auto-gptq autoawq datasets accelerate sentencepiece protobuf pandas tqdm

In [ ]:
import os
import gc
import time
import torch
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GPTQConfig
from datasets import load_dataset

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.mem_get_info(0)[1] / 1024**3:.1f} GB")

# Path to the merged fine-tuned model from notebook 02
MERGED_MODEL_PATH = "gemma2-2b-medical-merged"
RESULTS_FILE = "results/tables/all_results.csv"
os.makedirs("results/tables", exist_ok=True)

## Helper: Evaluation Functions

In [ ]:
BATCH_SIZE = 4  # Gemma 2 2B fits comfortably on T4

def evaluate_model(model, tokenizer, model_name, device="cuda"):
    """Run full evaluation suite and return results dict."""
    results = {"model": model_name}
    model.eval()

    # --- Perplexity: WikiText-2 ---
    print(f"  [1/5] WikiText-2 perplexity...")
    wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    wiki_texts = [t for t in wiki["text"] if len(t.strip()) > 50][:256]
    total_loss, total_tokens = 0.0, 0
    for i in tqdm(range(0, len(wiki_texts), BATCH_SIZE)):
        batch = wiki_texts[i:i+BATCH_SIZE]
        enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
        with torch.no_grad():
            out = model(**enc, labels=enc["input_ids"])
        n = enc["attention_mask"].sum().item()
        total_loss += out.loss.item() * n
        total_tokens += n
        del enc, out
    torch.cuda.empty_cache()
    results["perplexity_wikitext"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
    print(f"    -> {results['perplexity_wikitext']:.2f}")

    # --- Perplexity: Medical ---
    print(f"  [2/5] Medical text perplexity...")
    pqa = load_dataset("pubmed_qa", "pqa_labeled", split="train")
    med_texts = []
    for ex in pqa:
        ctx_data = ex.get("context", {})
        contexts = ctx_data.get("contexts", [])
        ctx = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
        if len(ctx.strip()) > 50:
            med_texts.append(ctx)
        if len(med_texts) >= 256:
            break
    total_loss, total_tokens = 0.0, 0
    for i in tqdm(range(0, len(med_texts), BATCH_SIZE)):
        batch = med_texts[i:i+BATCH_SIZE]
        enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
        with torch.no_grad():
            out = model(**enc, labels=enc["input_ids"])
        n = enc["attention_mask"].sum().item()
        total_loss += out.loss.item() * n
        total_tokens += n
        del enc, out
    torch.cuda.empty_cache()
    results["perplexity_medical"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
    print(f"    -> {results['perplexity_medical']:.2f}")

    # --- PubMedQA accuracy ---
    print(f"  [3/5] PubMedQA accuracy...")
    correct, total = 0, 0
    for ex in tqdm(pqa, total=min(len(pqa), 200), desc="PubMedQA", leave=False):
        if total >= 200:
            break
        ctx_data = ex.get("context", {})
        contexts = ctx_data.get("contexts", [])
        context = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
        prompt = (
            f"<start_of_turn>user\nContext: {context}\n\n"
            f"Question: {ex['question']}\n"
            f"Answer with exactly one word: yes, no, or maybe.<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
        resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
        pred = resp.split()[0].strip(".,!?;:") if resp.split() else ""
        gold = ex["final_decision"].lower().strip()
        if pred in ("yes", "no", "maybe") and pred == gold:
            correct += 1
        total += 1
        del inputs, out
        torch.cuda.empty_cache()
    results["pubmedqa_accuracy"] = correct / total if total else 0
    print(f"    -> {results['pubmedqa_accuracy']:.4f} ({correct}/{total})")

    # --- Inference speed ---
    print(f"  [4/5] Inference speed...")
    prompt = "<start_of_turn>user\nWhat is diabetes?<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=50, do_sample=False)
    total_tok, total_t = 0, 0.0
    for _ in range(5):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=50, do_sample=False)
        torch.cuda.synchronize()
        total_tok += out.shape[1] - inputs["input_ids"].shape[1]
        total_t += time.perf_counter() - t0
    results["tokens_per_sec"] = total_tok / total_t
    print(f"    -> {results['tokens_per_sec']:.1f} tok/s")

    # --- Peak VRAM ---
    print(f"  [5/5] Peak VRAM...")
    results["peak_vram_gb"] = torch.cuda.max_memory_allocated() / (1024**3)
    print(f"    -> {results['peak_vram_gb']:.2f} GB")

    return results


def save_results(results_dict):
    """Append or update results in the CSV file."""
    df_new = pd.DataFrame([results_dict])
    if os.path.exists(RESULTS_FILE):
        df = pd.read_csv(RESULTS_FILE)
        df = df[df["model"] != results_dict["model"]]
        df = pd.concat([df, df_new], ignore_index=True)
    else:
        df = df_new
    df.to_csv(RESULTS_FILE, index=False)
    return df

## 1. GPTQ Quantization (8-bit, 4-bit, 3-bit)

In [ ]:
# Prepare medical calibration data for GPTQ/AWQ
# Using 64 samples (not 128) to reduce memory pressure on T4
pqa_cal = load_dataset("pubmed_qa", "pqa_labeled", split="train")
calibration_texts = []
for ex in pqa_cal:
    ctx_data = ex.get("context", {})
    contexts = ctx_data.get("contexts", [])
    ctx = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    if len(ctx.strip()) > 50:
        calibration_texts.append(ctx)
    if len(calibration_texts) >= 64:
        break

print(f"Calibration samples: {len(calibration_texts)}")

In [ ]:
# GPTQ 8-bit
print("\n" + "="*60)
print("GPTQ 8-bit")
print("="*60)

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
gptq_config = GPTQConfig(bits=8, group_size=128, desc_act=True, dataset=calibration_texts, tokenizer=tokenizer)

model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH, quantization_config=gptq_config, device_map="auto", torch_dtype=torch.float16
)

torch.cuda.reset_peak_memory_stats()
res = evaluate_model(model, tokenizer, "gemma2-2b-med-gptq-8bit")
save_results(res)
del model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# GPTQ 4-bit
print("\n" + "="*60)
print("GPTQ 4-bit")
print("="*60)

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
gptq_config = GPTQConfig(bits=4, group_size=128, desc_act=True, dataset=calibration_texts, tokenizer=tokenizer)

model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH, quantization_config=gptq_config, device_map="auto", torch_dtype=torch.float16
)

torch.cuda.reset_peak_memory_stats()
res = evaluate_model(model, tokenizer, "gemma2-2b-med-gptq-4bit")
save_results(res)
del model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# GPTQ 3-bit
print("\n" + "="*60)
print("GPTQ 3-bit")
print("="*60)

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
gptq_config = GPTQConfig(bits=3, group_size=128, desc_act=True, dataset=calibration_texts, tokenizer=tokenizer)

model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH, quantization_config=gptq_config, device_map="auto", torch_dtype=torch.float16
)

torch.cuda.reset_peak_memory_stats()
res = evaluate_model(model, tokenizer, "gemma2-2b-med-gptq-3bit")
save_results(res)
del model; torch.cuda.empty_cache(); gc.collect()

## 2. AWQ Quantization (4-bit)

In [ ]:
print("\n" + "="*60)
print("AWQ 4-bit")
print("="*60)

from awq import AutoAWQForCausalLM

model = AutoAWQForCausalLM.from_pretrained(MERGED_MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)

quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
model.quantize(tokenizer, quant_config=quant_config, calib_data=calibration_texts)

# Save quantized model
os.makedirs("models/quantized/awq-4bit", exist_ok=True)
model.save_quantized("models/quantized/awq-4bit")
tokenizer.save_pretrained("models/quantized/awq-4bit")

# For evaluation, load with transformers
del model; torch.cuda.empty_cache(); gc.collect()

model = AutoModelForCausalLM.from_pretrained(
    "models/quantized/awq-4bit", device_map="auto", torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained("models/quantized/awq-4bit")

torch.cuda.reset_peak_memory_stats()
res = evaluate_model(model, tokenizer, "gemma2-2b-med-awq-4bit")
save_results(res)
del model; torch.cuda.empty_cache(); gc.collect()

## 3. BitsAndBytes NF4 (4-bit)

In [ ]:
print("\n" + "="*60)
print("BnB NF4 (4-bit)")
print("="*60)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH, quantization_config=bnb_config, device_map="auto"
)

torch.cuda.reset_peak_memory_stats()
res = evaluate_model(model, tokenizer, "gemma2-2b-med-bnb-nf4")
save_results(res)
del model; torch.cuda.empty_cache(); gc.collect()

## 4. BitsAndBytes INT8 (8-bit)

In [ ]:
print("\n" + "="*60)
print("BnB INT8 (8-bit)")
print("="*60)

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH, quantization_config=bnb_config, device_map="auto"
)

torch.cuda.reset_peak_memory_stats()
res = evaluate_model(model, tokenizer, "gemma2-2b-med-bnb-int8")
save_results(res)
del model; torch.cuda.empty_cache(); gc.collect()

## 5. View All Results

In [ ]:
df = pd.read_csv(RESULTS_FILE)
print("\nAll Results:")
print("="*100)
df